In [1]:
# !pip install nba_api

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nba_api.stats.endpoints import leaguedashteamstats
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings

warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


def fetch_team_data(seasons=None):
    """Fetch NBA team data from multiple seasons."""
    if seasons is None:
        seasons = [
            '2019-20',
            '2020-21',
            '2021-22',
            '2022-23',
            '2023-24',
            '2024-25'
        ]

    print(f"Fetching NBA team data for {len(seasons)} seasons...")

    all_dfs = []

    for season in seasons:
        print(f"  -> Fetching {season}")
        team_stats = leaguedashteamstats.LeagueDashTeamStats(
            season=season,
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Advanced'
        )
        df = team_stats.get_data_frames()[0]
        df['SEASON'] = season  # 👈 track season
        all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)

    print(f"Total rows collected: {len(combined_df)}")

    return combined_df


def prepare_data(df):
    """Prepare features and create target variable."""
    features = [
        'OFF_RATING',
        'DEF_RATING',
        'NET_RATING',
        'PACE',
        'TS_PCT',
        'EFG_PCT',
        'AST_RATIO',
        'REB_PCT',
        'TM_TOV_PCT',
        'PIE'
    ]

    df_clean = df[['TEAM_NAME'] + features + ['W_PCT']].dropna()
    df_clean['WIN_TIER'] = pd.qcut(
        df_clean['W_PCT'],
        3,
        labels=['Low', 'Mid', 'High'],
        duplicates='drop'
    )

    X = df_clean[features].copy()
    y = df_clean['WIN_TIER']
    team_names = df_clean['TEAM_NAME']

    print(f"Data shape: {X.shape}")
    print(f"Target distribution:\n{y.value_counts().sort_index()}")

    return X, y, team_names


def scale_features(X_train, X_test):
    """Standardize features for SVM."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled


def plot_labeled_confusion_matrix(y_true, y_pred, class_names, title, filename):
    """Plot and save a labeled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=class_names)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Saved: {filename}")


def run_svm_experiment(X_train, X_test, y_train, y_test, class_names, kernel, C, idx):
    """Run SVM with specific kernel and C value."""
    print(f"\n--- SVM: kernel={kernel}, C={C} ---")

    X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

    clf = SVC(kernel=kernel, C=C, random_state=42)
    clf.fit(X_train_scaled, y_train)

    y_pred = clf.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=class_names))

    plot_labeled_confusion_matrix(
        y_test, y_pred, class_names,
        f'SVM - {kernel.upper()} kernel, C={C}\nAccuracy: {acc:.4f}',
        f'outputs/svm_{kernel}_c{C}.png'
    )

    return acc


def main():
    df = fetch_team_data()
    X, y, team_names = prepare_data(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    class_names = y.unique()

    kernels = ['linear', 'rbf', 'poly']
    c_values = [0.1, 1.0, 10.0]

    results = {}

    for kernel in kernels:
        results[kernel] = {}
        for C in c_values:
            acc = run_svm_experiment(
                X_train, X_test, y_train, y_test,
                class_names, kernel, C, len(results[kernel])
            )
            results[kernel][C] = acc

    print("\n" + "="*60)
    print("SUMMARY: SVM Performance Across Kernels and C Values")
    print("="*60)

    best_per_kernel = {}
    for kernel in kernels:
        best_C = max(results[kernel], key=results[kernel].get)
        best_per_kernel[kernel] = (best_C, results[kernel][best_C])

    print(f"\n{'Kernel':<10} {'Best C':<10} {'Best Accuracy':<15}")
    print("-" * 35)
    for kernel in kernels:
        best_C, best_acc = best_per_kernel[kernel]
        print(f"{kernel:<10} {best_C:<10} {best_acc:.4f}")

    best_kernel = max(best_per_kernel, key=lambda k: best_per_kernel[k][1])
    best_C, best_acc = best_per_kernel[best_kernel]

    print(f"\n{'='*60}")
    print(f"BEST OVERALL: {best_kernel.upper()} kernel with C={best_C}")
    print(f"Accuracy: {best_acc:.4f}")
    print("="*60)


    print("\n=== Summary ===")
    print(f"All confusion matrices saved to /outputs folder:")
    for kernel in kernels:
        for C in c_values:
            print(f"  - svm_{kernel}_c{C}.png (accuracy: {results[kernel][C]:.4f})")


if __name__ == "__main__":
    main()


Fetching NBA team data for 6 seasons...
  -> Fetching 2019-20
  -> Fetching 2020-21
  -> Fetching 2021-22
  -> Fetching 2022-23
  -> Fetching 2023-24
  -> Fetching 2024-25
Total rows collected: 180
Data shape: (180, 10)
Target distribution:
WIN_TIER
Low     60
Mid     66
High    54
Name: count, dtype: int64

--- SVM: kernel=linear, C=0.1 ---
Accuracy: 0.8889
              precision    recall  f1-score   support

         Low       0.91      0.91      0.91        11
        High       1.00      0.83      0.91        12
         Mid       0.80      0.92      0.86        13

    accuracy                           0.89        36
   macro avg       0.90      0.89      0.89        36
weighted avg       0.90      0.89      0.89        36

Saved: outputs/svm_linear_c0.1.png

--- SVM: kernel=linear, C=1.0 ---
Accuracy: 0.9167
              precision    recall  f1-score   support

         Low       1.00      0.91      0.95        11
        High       0.92      0.92      0.92        12
        